In [2]:
import os
import camelot
import pandas as pd
from tqdm import tqdm
from IPython.display import display
import warnings
import re
import numpy as np
import pdfplumber

warnings.filterwarnings("ignore")

folder = os.getcwd()
pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


def extract_numbers(name):

    nums = re.findall(r'\d+', name)
    return tuple(map(int, nums))

replacements = {
    'Ⅰ': 'I',
    'Ⅱ': 'II',
    'Ⅲ': 'III',
    'Ⅳ': 'IV',
    'Ⅴ': 'V',
    'Ⅵ': 'VI',
    'Ⅶ': 'VII',
    'Ⅷ': 'VIII',
    'Ⅸ': 'IX',
    'Ⅹ': 'X'
}

def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):

    os.makedirs(out_dir, exist_ok=True)

    tables_lattice = camelot.read_pdf(
        pdf_path,
        pages=pages,
        flavor="lattice",
        line_scale=80
    )

    lattice_pages = set()
    text_pages = set()

    with pdfplumber.open(pdf_path) as pdf:

        for i, t in enumerate(tables_lattice):

            page_num = int(t.page)          
            page_idx = page_num - 1        
            lattice_pages.add(page_num)

            t.to_csv(
                os.path.join(out_dir, f"lattice_p{page_num}_{i}.csv"),
                index=False
            )

            page = pdf.pages[page_idx]

            x1, y1, x2, y2 = t._bbox
            page_w = page.width
            page_h = page.height

            table_top = page_h - y2

            left   = max(0, min(x1, page_w))
            right  = max(0, min(x2, page_w))
            top    = max(0, min(table_top - title_height, page_h))
            bottom = max(0, min(table_top, page_h))

            if right > left and bottom > top:
                title_box = (left, top, right, bottom)
                title_text = page.crop(title_box, strict=False).extract_text(
                    x_tolerance=2,
                    y_tolerance=2
                ) or ""

                if title_text.strip():
                    text_pages.add(page_num)
            else:
                title_text = ""

            with open(
                os.path.join(out_dir, f"text_p{page_num}_{i}.txt"),
                "w",
                encoding="utf-8"
            ) as f:
                f.write(title_text.strip())

    return None       


def join_ordered(series):
    vals = series.dropna().astype(str).str.strip()
    vals = [v for v in vals if v]
    seen = set()
    result = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return "; ".join(result) if result else None

def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if 'Qty' in df.columns:
        agg_map = {
            "Qty": "sum",
        }
    elif 'Код материалов' in df.columns:
        agg_map = {
            "Кол-во": "sum",
        }
    elif '件数' in df.columns:
        agg_map = {
            '件数': 'sum'
        }
    elif 'Units' in df.columns:
        agg_map = {
            'Units': 'sum'
        }

    if "Item and Spec" in df.columns:

        agg_map["Item and Spec"] = join_ordered
    if "Remark" in df.columns:
        agg_map["Remark"] = join_ordered

    if 'Наименование и спецификация' in df.columns:
        agg_map['Наименование и спецификация'] = join_ordered

    if 'Примечание' in df.columns:
        agg_map['Примечание'] = join_ordered

    if 'Версия' in df.columns:
        agg_map['Версия'] = join_ordered

    if 'Rev.' in df.columns:
        agg_map['Rev.'] = join_ordered

    if "名称及规格" in df.columns:
        agg_map["名称及规格"] = join_ordered

    if "备注" in df.columns:
        agg_map["备注"] = join_ordered

    if "Section" in df.columns:
        agg_map["Section"] = join_ordered

    if 'Material No.' in df.columns:
        df_agg = df.groupby("Material No.", as_index=False).agg(agg_map)
    elif 'Код материалов' in df.columns:
        df_agg = df.groupby("Код материалов", as_index=False).agg(agg_map)
    elif '物料编码' in df.columns:
        df_agg = df.groupby("物料编码", as_index=False).agg(agg_map)


    return df_agg

In [61]:
b = 36
e = b+1

for i, pth in enumerate(tqdm(pdf_files[b:e])):
    
    filename = os.path.basename(pth)          
    filename = os.path.splitext(filename)[0]
    print(f'Обработка файла: {filename}')

    # extract_tables_camelot(pth, filename, pages="all", title_height=800)

    dfs = []

    for file in sorted(os.listdir(filename), key=extract_numbers):

        if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
            continue

        path = os.path.join(filename, file)

        if os.path.getsize(path) == 0:  
            continue
        df_i = pd.read_csv(path, converters={1: str})

        if (
            df_i.empty
            or df_i.shape[1] < 4
            or not any(col in df_i.columns for col in ['NO.', 'Код материалов', 'No.', 'Код мат-ла', 'Serial No.', 'Part No.', '物料编码', 'п/п  Код материалов', 'Код'])
            ):
            continue

        # если китаец насрал пробел в начале таблицы
        if 'NO.' in df_i.iloc[0].values or 'Material No.' in df_i.iloc[0].values:
            df_i.columns = df_i.iloc[0]  
            df_i = df_i[1:].reset_index(drop=True) 

        for col in ['NO.', 'No.', '№ п/п', '№ \nп/п', 'Serial No.', '序号', '№ \n\nп/п', '№ \r\nп/п', '№']:
            if col in df_i.columns:
                df_i = df_i.drop(col, axis=1)

        idx = file.replace("lattice_", "").rsplit(".", 1)[0]
        section_val = os.path.join(filename, f"text_{idx}.txt")
        
        with open(section_val, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        skip_titles = [
            'Руководство по деталъная структура',
            'Альбом чертежей деталей и узлов коленчатого подъемника для высотных работ',
            'Альбом чертежей деталей и компонентов',
            'Parts Manual',
            '零部件图册'
        ]

        if lines and lines[0] in skip_titles:
            lines = lines[1:]
                
        if lines and lines[0] in ['Альбом чертежей деталей и узлов коленчатого', 'Альбом чертежей деталей и узлов коленчатого подъемника для']:
            lines = lines[2:]

        lines_joined = " ".join(lines)
            
        df_i["Section"] = lines_joined
        
        df_i = df_i[df_i.iloc[:, 0].replace('', pd.NA).notna()]
        display(df_i)
        df_i = df_i[df_i.iloc[:, 0] != '/']
        dfs.append(df_i)

    df_all = pd.concat(dfs, ignore_index=True)
    display(df_all)
    df_all.to_excel('dfs.xlsx', index=False)

    for old_col in ['во', 'Кол-\nво', 'Кол-\r\nво', 'Кол-в\nо \nштук', 'Кол-\nво \nшту\nк', 'Кол-во \nштук', 'Cantidad', 'Кол-во\n\nштук', 'Кол-во\r\nштук', 'Кол-\n\nво \n\nшту\n\nк', 'Кол-во \r\nштук', 'Кол-в\r\nо \r\nштук', 'Кол-\r\nво \r\nшту\r\nк', 'Кол-во штук', 'Кол-в', 'Колич\r\nество', 'Кол-в\r\nо\r\nштук']:
        if old_col in df_all.columns.tolist():
            if 'Кол-во' in df_all.columns.tolist():
                df_all['Кол-во'] = df_all.loc[:, old_col].combine_first(df_all['Кол-во'])
            else:
                df_all['Кол-во'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Кол-во' in df_all.columns:
        col = df_all.pop("Кол-во")  
        df_all.insert(2, "Кол-во", col) 

    for old_col in ['Наименование и \nспецификация', 'Наименование и \r\nспецификация', 'Наименование и\n\nспецификация', 'Nombre y especificación', 'Наименование и \r\nспецификация', 'Наименование и\r\nспецификация', 'Наименование и', 'Наименование\r\nи', 'Наименование']:
        if old_col in df_all.columns.tolist():
            if 'Наименование и спецификация' in df_all.columns:
                df_all['Наименование и спецификация'] = df_all.loc[:, old_col].combine_first(df_all['Наименование и спецификация'])
            else:
                df_all['Наименование и спецификация'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Наименование и спецификация' in df_all.columns:
        col = df_all.pop("Наименование и спецификация")  
        df_all.insert(1, "Наименование и спецификация", col) 

    for old_col in ['Name and Spec.', 'Spec and Item']:
        if old_col in df_all.columns:
            if 'Item and Spec' in df_all.columns:
                df_all['Item and Spec'] = df_all.loc[:, old_col].combine_first(df_all['Item and Spec'])
            else:
                df_all['Item and Spec'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Item and Spec' in df_all.columns:
        col = df_all.pop('Item and Spec')  
        df_all.insert(1, 'Item and Spec', col) 

    for old_col in ['Примечани\nе', 'Observaciones', 'Примечани\n\nе', 'Примечани\r\nе', '№ позиции', 'Примеч\r\nание']:
        if old_col in df_all.columns:
            if 'Примечание' in df_all.columns:
                df_all['Примечание'] = df_all.loc[:, old_col].combine_first(df_all['Примечание'])
            else:
                df_all['Примечание'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Примечание' in df_all.columns:
        col = df_all.pop('Примечание')  
        df_all.insert(3, 'Примечание', col) 

    for old_col in ['Qty.', 'Unit']:
        if old_col in df_all.columns:
            if 'Qty' in df_all.columns:
                df_all['Qty'] = df_all.loc[:, old_col].combine_first(df_all['Qty'])
            else:
                df_all['Qty'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Qty' in df_all.columns:
        col = df_all.pop('Qty')  
        df_all.insert(2, 'Qty', col) 
        
    for old_col in ['Código de material', 'Код мат-ла', 'п/п  Код материалов', 'Код']:
        if old_col in df_all.columns:
            if 'Код материалов' in df_all.columns:
                df_all['Код материалов'] = df_all.loc[:, old_col].combine_first(df_all['Код материалов'])
            else:
                df_all['Код материалов'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Код материалов' in df_all.columns:
        col = df_all.pop('Код материалов')  
        df_all.insert(0, 'Код материалов', col) 

    if 'Версия  Кол-' in df_all.columns:
        df_all['Версия'] = df_all.loc[:, 'Версия  Кол-'].combine_first(df_all['Версия'])
        df_all = df_all.drop(columns='Версия  Кол-')

    for old_col in ['Part No.']:
        if old_col in df_all.columns:
            if 'Material No.' in df_all.columns:
                df_all['Material No.'] = df_all.loc[:, old_col].combine_first(df_all['Material No.'])
            else:
                df_all['Material No.'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Material No.' in df_all.columns:
        col = df_all.pop('Material No.')  
        df_all.insert(0, 'Material No.', col) 

    for old_col in ['Note']:
        if old_col in df_all.columns:
            if 'Remark' in df_all.columns:
                df_all['Remark'] = df_all.loc[:, old_col].combine_first(df_all['Remark'])
            else:
                df_all['Remark'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Remark' in df_all.columns:
        col = df_all.pop('Remark')  
        df_all.insert(3, 'Remark', col) 

    for col in ['Кол-во', 'Qty', '件数']:
        if col in df_all.columns:
            df_all[col] = df_all[col].replace({'/': 0, '、': 0, 'A': 0})
            df_all[col] = df_all[col].fillna(0)
            df_all[col] = df_all[col].astype(int)
    
    df_all.to_excel('df_all.xlsx', index=False)
    df_all = pd.read_excel('df_all.xlsx')
    test_1 = pd.read_excel('dfs.xlsx')
    assert test_1.shape[0] == df_all.shape[0]

    for col in ['Material No.', 'Код материалов', '物料编码']:
        if col in df_all.columns:
            df_all = df_all[df_all[col] != '']
            df_all = df_all[~df_all[col].isna()]

    display(df_all.head())

    if 'Material No.' in df_all.columns and df_all['Material No.'].astype(str).str.contains('NO-NUMBER').any():
        df_all['Material No.'] = df_all['Material No.'].str.split('\n').str[0]
        df_all = df_all[~df_all['Material No.'].astype(str).str.startswith('NO-NUMBER')]

    if 'Код материалов' in df_all.columns and df_all['Код материалов'].astype(str).str.contains('NO-NUMBER').any():
        df_all = df_all[~df_all['Код материалов'].astype(str).str.startswith('NO-NUMBER')]

    cols = df_all.select_dtypes(include='object').columns
    df_all[cols] = (df_all[cols].replace(r'\s*\n\s*', ' ', regex=True))
    df_all[cols] = df_all[cols].replace(replacements, regex=True)

    if 'Item and Spec' in df_all.columns:
        df_all['Item and Spec'] = df_all['Item and Spec'].replace('/', np.nan)
    
    if '物料编码' in df_all.columns:
        mask = df_all['物料编码'].str.contains(r'^\d+\s+\D', na=False)
        extracted = df_all.loc[mask, '物料编码'].str.extract(r'^(\d+)\s+(.*)$')
        df_all.loc[mask, '物料编码'] = extracted[0]
        df_all.loc[mask, '名称及规格'] = extracted[1]

    df_merged = merge_parts(df_all) 
   
    assert (df_all.shape[0] - df_all.iloc[:, 0].duplicated().sum() - df_merged.shape[0]) == 0

    lst = ['ZS1623RT', 'ZA10RJE', 'ZA14J', 'ZA14JE-Li', 'ZA14NJE', 'ZA16J', 'ZA16JERT', 'ZA18J', 'ZA20J', 'ZA20JE', 'ZA20JERT', 'ZA22J-V' ,'ZA22J', 'ZA24J', 'ZA32J(H-образ.)', 'ZA32J(X-образ.)', 'ZT42J-V', 'ZA42J', '', '', '', '', '', '', '', '', '' ,'', '', 'ZT20J', 'ZT22J-V', 'ZT26J', 'ZT26JS-V', 'ZT30J', 'ZT32J-V', 'ZT32J', 'ZT34J']
    df_merged['Models'] = lst[b]

    df_merged.to_excel(f'{filename}.xlsx', index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: ZT34J


,Код мат-ла,Наименование и\r\nспецификация,Кол-во\r\nштук,Примечание,Section
0,00773406200210000,Электрический шкаф\r\nуправления платформой в\...,1,NaN,Рабочая платформа00771100100000000
1,1040200503,Гайка M8-8,6,GB/T 889.1-2000,Рабочая платформа00771100100000000
2,1040302480,Шайба 8-200HV,12,GB/T 96.1-2002,Рабочая платформа00771100100000000
3,00773406300201030,Резиновая прокладка,2,NaN,Рабочая платформа00771100100000000
4,1040004619,Болт M8×35-8.8,4,GB/T 5783-2000,Рабочая платформа00771100100000000
5,1040201896,Гайка M6-8,13,GB/T 889.1-2000,Рабочая платформа00771100100000000
6,1040300771,Шайба 6-200HV,23,GB/T 97.1-2002,Рабочая платформа00771100100000000
7,00773400100261110,Упор,2,NaN,Рабочая платформа00771100100000000
8,1040006017,Винт M6×25-A2-70,9,GB/T 70.1-2000,Рабочая платформа00771100100000000
9,00773400100260020,Защитный кожух,1,NaN,Рабочая платформа00771100100000000


,Код мат-ла,Наименование и\r\nспецификация,Кол-во\r\nштук,Примечание,Section
0,1040302490,Шайба 12-200HV,16,GB/T 97.1-2002,Рабочая платформа00771100100000000
1,00771600100001040,Регулировочная прокладка\r\nкачающегося цилиндра,1,NaN,Рабочая платформа00771100100000000
2,1040301511,Шайба 24-200HV,3,GB/T 97.1-2002,Рабочая платформа00771100100000000
3,1040201111,Гайка M24-10,1,GB/T 889.1-2000,Рабочая платформа00771100100000000
4,1040302821,Шайба 12-200HV,2,GB/T 96.1-2002,Рабочая платформа00771100100000000
5,00771600100200000,Узел платформы,1,NaN,Рабочая платформа00771100100000000
6,00771600100400000,Узел кронштейна,1,NaN,Рабочая платформа00771100100000000
7,1040102372,Винт M4×20-A2-70,2,GB/T 70.1-2000,Рабочая платформа00771100100000000
8,00771600101400000,Защитная крышка\r\nгидравлического клапана\r\n...,1,NaN,Рабочая платформа00771100100000000
9,00771600101600000,Сварная конструкция рамы\r\nгидравлического кл...,1,NaN,Рабочая платформа00771100100000000


,Код мат-ла,Наименование и\r\nспецификация,Кол-во\r\nштук,Примечание,Section
0,00773400100230000,Вращающаяся дверь,1,NaN,Узел платформы 00771600100200000
1,1101000138,Возвратный шарнир,2,NaN,Узел платформы 00771600100200000
2,1040006016,Винт M6×10-A2-70,16,GB/T 70.3-2000,Узел платформы 00771600100200000
3,00771600100210000,Сварная конструкция\r\nплатформы,1,NaN,Узел платформы 00771600100200000
4,00773400100201020,Сердцевина замка,1,NaN,Узел платформы 00771600100200000
5,00775600101801020,Пружина,1,NaN,Узел платформы 00771600100200000
6,1040004522,Болт,1,GB/T 5783-2000,Узел платформы 00771600100200000
7,1040301515,Шайба 8-200HV,2,GB/T 97.1-2002,Узел платформы 00771600100200000
8,1040201880,Гайка M8-6,1,GB/T 802-1988,Узел платформы 00771600100200000
9,1040201902,Гайка M6-6,12,GB/T 802-1988,Узел платформы 00771600100200000


...

,Код мат-ла,Наименование и\r\nспецификация,Кол-во\r\nштук,Примечание,Section
0,00771407000201010,"Опасность опрокидывания,\r\nзнак сигнализации ...",1,NaN,Маркировка в сборе 00771607000000000
1,00771607000201070,Знак инструкции управления\r\nпультом управлен...,1,NaN,Маркировка в сборе 00771607000000000
2,00771407000201130,LOGO ЗУМЛИОН стрелы,1,NaN,Маркировка в сборе 00771607000000000
3,00773407000201010,LOGO ЗУМЛИОН платформы,1,NaN,Маркировка в сборе 00771607000000000
4,00773407000201050,ЗУМЛИОН противовеса,1,NaN,Маркировка в сборе 00771607000000000
5,00771407000201160,Гидравлический бак,1,NaN,Маркировка в сборе 00771607000000000
6,00771607000201020,Знак схемы безопасного\r\nрабочего диапазона,1,NaN,Маркировка в сборе 00771607000000000
7,00771607000201030,Знак «Опасность\r\nопрокидывания»,1,NaN,Маркировка в сборе 00771607000000000
8,00771607000201040,Номинальное значение уклона,1,NaN,Маркировка в сборе 00771607000000000
9,00771607000201050,Знак ZT34J,1,NaN,Маркировка в сборе 00771607000000000


,Код мат-ла,Наименование и\r\nспецификация,Кол-во\r\nштук,Примечание,Section
0,00773407000201180,Знак «Запрещается наступать»,4,NaN,Маркировка в сборе 00771607000000000
1,00773407000201200,Знак «Запрещается\r\nподдерживать руками или\r...,3,NaN,Маркировка в сборе 00771607000000000
2,00773407000201230,Знак «Отказ педали»,1,NaN,Маркировка в сборе 00771607000000000
3,00773407000201470,Знак предупреждения ожогов,1,NaN,Маркировка в сборе 00771607000000000
4,00771407000201080,Знак в виде синей стрелки,1,NaN,Маркировка в сборе 00771607000000000
5,00771407000201090,Знак в виде желтой стрелки,1,NaN,Маркировка в сборе 00771607000000000
6,00771407000201100,Знак в виде желтого\r\nтреугольника,2,NaN,Маркировка в сборе 00771607000000000
7,00771407000201110,Знак в виде синего\r\nтреугольника,2,NaN,Маркировка в сборе 00771607000000000


,Код мат-ла,Наименование и\r\nспецификация,Кол-во\r\nштук,Примечание,Section,Наименование и спецификация,Кол-в\r\nо\r\nштук,№ п\r\n/п,Примеч\r\nание
0,00773406200210000,Электрический шкаф\r\nуправления платформой в\...,1,NaN,Рабочая платформа00771100100000000,NaN,NaN,NaN,NaN
1,1040200503,Гайка M8-8,6,GB/T 889.1-2000,Рабочая платформа00771100100000000,NaN,NaN,NaN,NaN
2,1040302480,Шайба 8-200HV,12,GB/T 96.1-2002,Рабочая платформа00771100100000000,NaN,NaN,NaN,NaN
3,00773406300201030,Резиновая прокладка,2,NaN,Рабочая платформа00771100100000000,NaN,NaN,NaN,NaN
4,1040004619,Болт M8×35-8.8,4,GB/T 5783-2000,Рабочая платформа00771100100000000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1269,00773407000201470,Знак предупреждения ожогов,1,NaN,Маркировка в сборе 00771607000000000,NaN,NaN,NaN,NaN
1270,00771407000201080,Знак в виде синей стрелки,1,NaN,Маркировка в сборе 00771607000000000,NaN,NaN,NaN,NaN
1271,00771407000201090,Знак в виде желтой стрелки,1,NaN,Маркировка в сборе 00771607000000000,NaN,NaN,NaN,NaN
1272,00771407000201100,Знак в виде желтого\r\nтреугольника,2,NaN,Маркировка в сборе 00771607000000000,NaN,NaN,NaN,NaN


,Код материалов,Наименование и спецификация,Кол-во,Примечание,Section,№ п\n\n/п
0,00773406200210000,Электрический шкаф\n\nуправления платформой в\...,1,NaN,Рабочая платформа00771100100000000,NaN
1,1040200503,Гайка M8-8,6,GB/T 889.1-2000,Рабочая платформа00771100100000000,NaN
2,1040302480,Шайба 8-200HV,12,GB/T 96.1-2002,Рабочая платформа00771100100000000,NaN
3,00773406300201030,Резиновая прокладка,2,NaN,Рабочая платформа00771100100000000,NaN
4,1040004619,Болт M8×35-8.8,4,GB/T 5783-2000,Рабочая платформа00771100100000000,NaN


100%|██████████| 1/1 [00:02<00:00,  2.82s/it]


- Спросить у Миши про запчасти, в описании которых встречается ещё одна модель, помимо основной, в которой они встречаются
- Что делать с такими в описании (Cubota): ZT138J-ZT24JE-V Turntable Decal Assembly - что значит тире?